⚠️ **Under construction.** <!-- banner:under-construction --> This lab is drafted but not yet instructor-reviewed; it may change without notice until its session.

# Lab 5 · Counting your way to a p-value

**Today:** when you walk out you can compute a p-value with nothing but a loop and a counter — because that is all a p-value is.

**Before you start:** chapter 2 read — this lab is its central computation with the biology removed. Plain coins only.

Each section names one idea, explains what it does, and asks you to **predict
what a cell prints before you run it**. Write the prediction down — on paper,
out loud, in a comment. A wrong prediction you wrote down teaches you exactly
one thing; a shrug teaches you nothing.

Most sections end with a **Test your understanding** task: write a small piece
of code, then run the check cell under it. The check never grades and never
breaks anything — ⬜ means not attempted yet, ❌ means not yet (with a hint),
✅ means passing. The check cells are the one thing here to run rather than
edit; everything else is yours to break.

**AI in this lab:** until your prediction is written down, work at level 1 — no
AI. The prediction is how you find out what you, unaided, can already read, and
both exams are level 1. Once you have run a cell, level 3 is encouraged: ask
your tutor to explain any miss.

Run every cell. Change things. Breaking this notebook costs nothing and teaches
more than reading it.

## 1 · An observation, and a question

You flipped a coin 30 times and got **22 heads**. Suspicious — but *how* suspicious? The honest question is chapter 2's: **if the coin were fair, how often would 30 flips look at least this extreme?** Nothing below answers anything else.

First, the world where the answer is known — the fair coin, simulated. Predict nothing yet; just read the function against lab 3's `count_heads`: it is the same function.

In [ ]:
import numpy as np

def count_heads(n_flips, chance, rng):
    heads = 0
    for _ in range(n_flips):
        if rng.random() < chance:
            heads += 1
    return heads

rng = np.random.default_rng(37)
print("five fair-coin runs of 30:",
      count_heads(30, 0.5, rng), count_heads(30, 0.5, rng),
      count_heads(30, 0.5, rng), count_heads(30, 0.5, rng),
      count_heads(30, 0.5, rng))

Five ordinary fair-coin mornings. None reached 22 — but five runs is nothing. The question needs thousands, and a counter.

**Test your understanding.** Simulate 5,000 fair-coin runs of 30 flips (generator: seed 41, one `count_heads` call per run, in a plain loop) and count how many runs came out **at 22 heads or more**. Name the counter `n_extreme`.

In [ ]:
# your turn: n_extreme — of 5000 fair runs (seed 41), how many reached 22+ heads?
import numpy as np
rng = np.random.default_rng(41)

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("n_extreme", expect=50,
      hint="one count_heads(30, 0.5, rng) per run, all 5000 from the seed-41 generator; count runs where the result >= 22")

## 2 · The fraction is the answer

Your counter divided by the number of runs is the fraction of fair-coin worlds at least as extreme as what you saw. Chapter 2 adds one small honesty correction — count your own observation as one more extreme world — so the recipe is:

    p = (n_extreme + 1) / (n_runs + 1)

That is a **p-value**. Not the probability the coin is fair; the frequency with which fairness produces data like yours. Run the arithmetic on your section-1 counter — predict its rough size first (50-ish out of 5,000 is about what percent?).

In [ ]:
print("p =", (50 + 1) / (5000 + 1))

About 0.0102 — one fair morning in a hundred looks like yours. Evidence, and now you know exactly where it came from: a loop, an `if`, a counter, a division. Every p-value you will ever meet is this computation, however it was dressed.

**Test your understanding.** Wrap it: `p_value(observed, n_flips, n_runs, rng)` returning the pseudocounted fraction of fair runs with `observed` or more heads.

In [ ]:
# your turn: p_value(observed, n_flips, n_runs, rng)
import numpy as np

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("p_value", expect=1.0, args=(0, 30, 200, np.random.default_rng(7)),
      hint="every run has 0 or more heads — the fraction is forced")
check("p_value", expect=0.0102, args=(22, 30, 5000, np.random.default_rng(41)),
      hint="this should reproduce sections 1-2 exactly: same seed, same draws, "
           "then round to 4 decimals — or return (n+1)/(runs+1) and compare")

## 3 · Twenty coins at once

Chapter 2's warning arrives when you test *many* coins. Twenty perfectly fair coins, 30 flips each, and report every coin whose p-value lands under 0.05. **Predict: how many of these twenty fair coins get accused?** Zero is a popular wrong answer.

In [ ]:
import numpy as np

def count_heads(n_flips, chance, rng):
    heads = 0
    for _ in range(n_flips):
        if rng.random() < chance:
            heads += 1
    return heads

def p_value(observed, n_flips, n_runs, rng):
    n_extreme = 0
    for _ in range(n_runs):
        if count_heads(n_flips, 0.5, rng) >= observed:
            n_extreme += 1
    return (n_extreme + 1) / (n_runs + 1)

rng = np.random.default_rng(43)
accused = 0
for coin in range(20):
    observed = count_heads(30, 0.5, rng)   # a FAIR coin's real morning
    p = p_value(observed, 30, 500, rng)
    if p < 0.05:
        accused += 1
print("fair coins accused:", accused, "of 20")

One fair coin accused (your run may differ — rerun with other seeds and watch it wobble around one). Nothing malfunctioned: a 0.05 threshold accuses about 5% of innocent coins *by design*, and twenty tests give it twenty chances. That arithmetic — how many accusations a threshold hands out for free — is chapter 2's second half, and week 9 builds the machinery that manages it at genome scale.

**Test your understanding.** One structural check to write yourself: `expected_accusations(n_tests, threshold)` — the number of innocent accusations a threshold should produce on average. No randomness needed; it is one multiplication.

In [ ]:
# your turn: expected_accusations(n_tests, threshold)

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("expected_accusations", expect=1.0, args=(20, 0.05))
check("expected_accusations", expect=400.0, args=(8000, 0.05),
      hint="the number the textbook calls the E[FP] bill")

## If you finish early

- Change section 3's threshold to 0.01 and predict the accusation count before rerunning.
- Chapter 2's practice problems run this same machinery on the real scan.

## If you are stuck

Wave someone over — this hour exists so a stuck step costs you a minute rather
than an evening. Known snags:

- **`n_extreme` misses the expected 50** — the check needs the exact seed-41 stream: create the generator once, then 5,000 `count_heads` calls and nothing else touching it.
- **`p_value` returns something near but not exactly 0.0102** — the check wants 4-decimal rounding; `round(x, 4)`.
- **Section 3 accuses far more than one** — check that each coin's `observed` comes from `chance=0.5`; a typo there makes guilty coins.